In [4]:
# Cleaning challenges intentionally injected:
# 1. Multiple date hashtags
# 2. Missing or ranged prices
# 3. Route expressed with multiple delimiters
# 4. Light typos and informal language

In [5]:
import random
from datetime import date, timedelta
import pandas as pd
import os
import re

SEED = 256
random.seed(SEED)

datasets = [
    (500,  "sample_posts_500.csv"),
    (2000, "sample_posts_2000.csv"),
    (5000, "sample_posts_5000.csv"),
]

OUT_DIR = "synthetic_data"
os.makedirs(OUT_DIR, exist_ok=True)

NOISE = {
    "multi_date_tag_prob": 0.10,
    "no_hashtag_prob":     0.06,
    "price_range_prob":    0.18,
    "price_text_prob":     0.12,
    "missing_price_prob":  0.14,
    "typo_prob":           0.10,
    "abbr_prob":           0.12,
}

DATE_START = date(2024, 1, 1)
DATE_END   = date(2025, 12, 31)

CITIES = ["Davis", "Sacramento", "San Francisco", "San Jose", "Oakland", "Berkeley", "Palo Alto", "Sunnyvale", "Fremont"]
CITY_ABBR = {"San Francisco": "SF", "San Jose": "SJ", "Sacramento": "Sac"}



In [6]:
def maybe(p): 
    return random.random() < p

def rand_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

def fmt_mmddyyyy(d):
    return d.strftime("%m%d%Y")

def make_hashtag(ride_or_drive, d):
    tag = f"#{ride_or_drive}{fmt_mmddyyyy(d)}"
    if maybe(0.08): tag = tag.lower()
    if maybe(0.06): tag = " " + tag
    return tag

def ride_drive_tag(ride_or_drive, d):
    if maybe(NOISE["no_hashtag_prob"]):
        return ""
    if maybe(NOISE["multi_date_tag_prob"]):
        d2 = d + timedelta(days=random.choice([1, 2, 3, 7]))
        return f"{make_hashtag(ride_or_drive, d)} {make_hashtag(ride_or_drive, d2)}"
    return make_hashtag(ride_or_drive, d)

def sample_price():
    if maybe(NOISE["missing_price_prob"]):
        return ""
    base = random.choice([10, 12, 15, 18, 20, 22, 25, 30, 35, 40])
    if maybe(NOISE["price_range_prob"]):
        hi = base + random.choice([5, 8, 10])
        return f"${base}-{hi}"
    if maybe(NOISE["price_text_prob"]):
        return f"{base} dollars"
    return f"${base}"

def choose_city():
    c = random.choice(CITIES)
    if c in CITY_ABBR and maybe(NOISE["abbr_prob"]):
        return CITY_ABBR[c]
    return c

def choose_route():
    a = choose_city()
    b = choose_city()
    while b == a:
        b = choose_city()
    return a, b

ROUTE_PATTERNS = [
    lambda a, b: f"{a} -> {b}",
    lambda a, b: f"{a} to {b}",
    lambda a, b: f"{a}-{b}",
    lambda a, b: f"{a} ➜ {b}",
    lambda a, b: f"from {a} to {b}",
]

def add_typos(s, prob):
    if not s or not maybe(prob) or len(s) < 5:
        return s
    i = random.randint(0, len(s) - 2)
    return s[:i] + s[i+1] + s[i] + s[i+2:]

def normalize_whitespace(s):
    if maybe(0.20):
        s = re.sub(r"\s+", " ", s)
    if maybe(0.15):
        s = s.replace(" ", "  ")
    return s

def sample_time_str():
    hr = random.choice(list(range(6, 23)))
    minute = random.choice([0, 10, 15, 20, 30, 40, 45, 50])
    ampm = "AM" if hr < 12 else "PM"
    hr12 = hr if 1 <= hr <= 12 else hr - 12
    if hr12 == 0: hr12 = 12
    return f"{hr12}:{minute:02d} {ampm}" if maybe(0.5) else f"{hr12}{ampm}"

def generate_post_text():
    ride_or_drive = random.choice(["ride", "drive"])
    d = rand_date(DATE_START, DATE_END)

    tag_part = ride_drive_tag(ride_or_drive, d)
    start, dest = choose_route()
    route_part = random.choice(ROUTE_PATTERNS)(start, dest)

    price = sample_price()
    tstr = sample_time_str()

    templates = [
        lambda: f"{tag_part} {route_part} {price} leaving {tstr}",
        lambda: f"{tag_part} Need a {ride_or_drive} {route_part}. Budget {price}",
        lambda: f"{tag_part} {ride_or_drive.upper()}!! {route_part} {tstr} {price}",
        lambda: f"{tag_part} {route_part} | {tstr} | {price} | 1 seat",
    ]
    text = random.choice(templates)().strip()

    if maybe(0.25):
        text += random.choice([" 😊", " 🚗", " pls", " thx", " !!"])
    if maybe(0.12):
        text += "\n" + random.choice(["comment below", "pm me", "anyone?", "flexible time"])

    text = add_typos(text, prob=NOISE["typo_prob"])
    text = normalize_whitespace(text)
    return text

# --- BA-style loop: generate + save + validate ---
for n, filename in datasets:
    rows = [{"post_id": i+1, "post_text": generate_post_text()} for i in range(n)]
    df = pd.DataFrame(rows)

    out_path = os.path.join(OUT_DIR, filename)
    df.to_csv(out_path, index=False)

    if os.path.exists(out_path):
        df_check = pd.read_csv(out_path)
        if len(df_check) == n:
            print(f"Success {filename}: {n} posts")
        else:
            print(f"Fail {filename}: expected {n}, got {len(df_check)}")
    else:
        print(f"Fail {filename}: File not created")

Success sample_posts_500.csv: 500 posts
Success sample_posts_2000.csv: 2000 posts
Success sample_posts_5000.csv: 5000 posts


In [7]:
def sample_time_str():
    hr = random.choice(list(range(6, 23)))
    minute = random.choice([0, 10, 15, 20, 30, 40, 45, 50])
    ampm = "AM" if hr < 12 else "PM"
    hr12 = hr if 1 <= hr <= 12 else hr - 12
    if hr12 == 0: hr12 = 12
    return f"{hr12}:{minute:02d} {ampm}" if maybe(0.5) else f"{hr12}{ampm}"

def generate_post_text():
    ride_or_drive = random.choice(["ride", "drive"])
    d = rand_date(DATE_START, DATE_END)

    tag_part = ride_drive_tag(ride_or_drive, d)  # may be empty
    start, dest = choose_route()
    route_part = random.choice(ROUTE_PATTERNS)(start, dest)

    price = sample_price()
    tstr = sample_time_str()

    templates = [
        lambda: f"{tag_part} {route_part} {price} leaving {tstr}",
        lambda: f"{tag_part} Need a {ride_or_drive} {route_part}. Budget {price}",
        lambda: f"{tag_part} {ride_or_drive.upper()}!! {route_part} {tstr} {price}",
        lambda: f"{tag_part} {route_part} | {tstr} | {price} | 1 seat",
    ]

    text = random.choice(templates)().strip()

    # generic noise
    if maybe(0.25):
        text += random.choice([" 😊", " 🚗", " pls", " thx", " !!"])
    if maybe(0.12):
        text += "\n" + random.choice(["comment below", "pm me", "anyone?", "flexible time"])

    # typo + whitespace noise
    text = add_typos(text, prob=NOISE["typo_prob"])
    text = normalize_whitespace(text)
    return text


In [8]:
for n, filename in datasets:
    rows = []

    for i in range(n):
        rows.append({
            "post_id": i + 1,
            "post_text": generate_post_text()
        })

    df = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, filename)
    df.to_csv(out_path, index=False)

    # validating
    if os.path.exists(out_path):
        df_check = pd.read_csv(out_path)
        if len(df_check) == n:
            print(f"Success {filename}: {n} posts")
        else:
            print(f"Fail {filename}: expected {n}, got {len(df_check)}")
    else:
        print(f"Fail {filename}: File not created")


Success sample_posts_500.csv: 500 posts
Success sample_posts_2000.csv: 2000 posts
Success sample_posts_5000.csv: 5000 posts


In [11]:
import pandas as pd
import re

df2k = pd.read_csv("synthetic_data/sample_posts_2000.csv")

# hashtag coverage + multi-tag
tag_re = re.compile(r"#\s*(ride|drive)\s*\d{8}", re.IGNORECASE)
tags_per_post = df2k["post_text"].apply(lambda s: len(tag_re.findall(s)) if isinstance(s, str) else 0)

# price coverage
has_dollar = df2k["post_text"].str.contains(r"\$", regex=True)
has_text_dollars = df2k["post_text"].str.contains(r"\b\d+\s*dollars\b", case=False, regex=True)

# route delimiter coverage
route_arrow = df2k["post_text"].str.contains(r"(?:->|→|➜)", regex=True)
route_to = df2k["post_text"].str.contains(r"\bto\b", case=False, regex=True)
route_dash = df2k["post_text"].str.contains(r"\w-\w", regex=True)

print("Rows:", len(df2k))
print("Hashtag posts (%):", (tags_per_post > 0).mean())
print("Multi-hashtag posts (%):", (tags_per_post > 1).mean())
print("Has $ price (%):", has_dollar.mean())
print("Has 'dollars' text price (%):", has_text_dollars.mean())
print("Has arrow route (%):", route_arrow.mean())
print("Has 'to' route (%):", route_to.mean())
print("Has dash route (%):", route_dash.mean())


Rows: 2000
Hashtag posts (%): 0.921
Multi-hashtag posts (%): 0.1055
Has $ price (%): 0.764
Has 'dollars' text price (%): 0.0815
Has arrow route (%): 0.4115
Has 'to' route (%): 0.381
Has dash route (%): 0.314
